# Data Exploration — P&L Restatement Case Study

This notebook is a first look at the five sample CSVs supplied with the assignment
(`docs/assignment.md`). The goal here is purely descriptive: understand exactly
what each file contains, its columns, its grain, and how the files relate to one
another — before designing the storage structure and restatement logic.

**Files, in the order we'll walk through them:**

1. `positions.csv` — the universe of positions.
2. `daily_pnl.csv` — the daily P&L as originally recorded.
3. `trade_detail.csv` — the trade-level inputs (quantity, execution price, closing price) behind one corrected trade.
4. `pnl_corrections.csv` — corrected execution prices, with receipt timestamps.
5. `period_status.csv` — when each month was closed.


In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 120)

DATA_DIR = Path("../data")

[p.name for p in sorted(DATA_DIR.glob("*.csv"))]


['daily_pnl.csv',
 'period_status.csv',
 'pnl_corrections.csv',
 'positions.csv',
 'trade_detail.csv']

## 1. `positions.csv` — the position universe

**Grain:** one row per position. This is the master/reference list; it never
changes over time in this sample (no `recorded_at` or effective-dating).

| Column | Type | Meaning |
|---|---|---|
| `position_id` | string, PK | Short code identifying the position, e.g. `P001`. Used as the join key everywhere else. |
| `security_name` | string | Human-readable name of the security/company, e.g. `Alpha Corp`. |
| `quantity` | number | Number of units held in this position. Used in the P&L formula `quantity × (closing_price − execution_price)`. |

There are exactly **3 positions** in the sample, which together make up "the whole fund" referenced in the assignment's questions.


In [2]:
positions = pd.read_csv(DATA_DIR / "positions.csv")
print(positions.dtypes, end="\n\n")
positions


position_id        str
security_name      str
quantity         int64
dtype: object



,position_id,security_name,quantity
0,P001,Alpha Corp,1000
1,P002,Beta Holdings,2500
2,P003,Gamma Industries,800


## 2. `daily_pnl.csv` — daily P&L as originally recorded

**Grain:** one row per `(position_id, business_date)` — the daily figure as it
was first booked, **before** any correction. The assignment calls this "daily
profit and loss as originally recorded"; these are the figures investors were
originally told about, day by day.

| Column | Type | Meaning |
|---|---|---|
| `position_id` | string, FK → `positions.position_id` | Which position the figure belongs to. |
| `business_date` | date | The trading day the figure is for. |
| `pnl_amount` | number | The P&L for that position on that day, as originally recorded (can be positive or negative). |
| `recorded_at` | timestamp | When this row was recorded — here, always the evening of the same `business_date` (18:00). This is the "as of" timestamp for the *original* figure, which matters once we start asking "what was known as of date D". |

Covers **all three positions**, every business day from **2025-01-01 through
2025-03-31** (weekdays only — no weekend rows). This is the full history of
originally-reported daily figures; it is **never edited** once written. One of
these rows (P001, 2025-01-15) will later be found to be wrong and superseded by
a correction — but the original row itself still remains in the data, untouched.


In [3]:
daily_pnl = pd.read_csv(DATA_DIR / "daily_pnl.csv", parse_dates=["business_date", "recorded_at"])
print(daily_pnl.dtypes, end="\n\n")
print("row count:", len(daily_pnl))
print("positions covered:", sorted(daily_pnl["position_id"].unique()))
print("date range:", daily_pnl["business_date"].min().date(), "to", daily_pnl["business_date"].max().date())
print("rows per position:\n", daily_pnl["position_id"].value_counts())
daily_pnl.head()


position_id                 str
business_date    datetime64[us]
pnl_amount                int64
recorded_at      datetime64[us]
dtype: object

row count: 192
positions covered: ['P001', 'P002', 'P003']
date range: 2025-01-01 to 2025-03-31
rows per position:
 position_id
P001    64
P002    64
P003    64
Name: count, dtype: int64


,position_id,business_date,pnl_amount,recorded_at
0,P001,2025-01-01,60,2025-01-01 18:00:00
1,P001,2025-01-02,110,2025-01-02 18:00:00
2,P001,2025-01-03,80,2025-01-03 18:00:00
3,P001,2025-01-06,130,2025-01-06 18:00:00
4,P001,2025-01-07,70,2025-01-07 18:00:00


The row we'll be tracking closely throughout the case study is **P001 (Alpha
Corp), 2025-01-15** — the one day whose original figure turns out to be wrong
and gets corrected (see §4 below).


In [4]:
daily_pnl[(daily_pnl["position_id"] == "P001") & (daily_pnl["business_date"] == "2025-01-15")]


,position_id,business_date,pnl_amount,recorded_at
10,P001,2025-01-15,1000,2025-01-15 18:00:00


## 3. `trade_detail.csv` — the trade behind the corrected figure

**Grain:** one row per `(position_id, business_date)` trade that we have
trade-level detail for. The assignment is explicit that this file covers
"the one trade you will see corrected" — so, unlike `daily_pnl.csv`, this file
is *not* a complete daily history; it only exists to let us recompute P&L from
first principles (`quantity × (closing_price − execution_price)`) for the one
day that gets corrected.

| Column | Type | Meaning |
|---|---|---|
| `position_id` | string, FK → `positions.position_id` | Which position. |
| `business_date` | date | The trading day. |
| `quantity` | number | Units traded (matches `positions.quantity` for this position in the sample). |
| `execution_price` | number | The price the trade was originally executed at. |
| `closing_price` | number | The closing market price used to mark the position at day-end. |

**Sanity check against the formula:** for P001 on 2025-01-15, the original
`daily_pnl.csv` figure was `1000`. Using the formula with the original
execution price here: `1000 × (50.00 − 49.00) = 1000` ✓ — confirms this trade
is exactly what produced that day's original (and, as we'll see, incorrect)
P&L figure.


In [5]:
trade_detail = pd.read_csv(DATA_DIR / "trade_detail.csv", parse_dates=["business_date"])
print(trade_detail.dtypes, end="\n\n")
trade_detail["implied_pnl"] = trade_detail["quantity"] * (trade_detail["closing_price"] - trade_detail["execution_price"])
trade_detail


position_id                   str
business_date      datetime64[us]
quantity                    int64
execution_price           float64
closing_price             float64
dtype: object



,position_id,business_date,quantity,execution_price,closing_price,implied_pnl
0,P001,2025-01-15,1000,49.0,50.0,1000.0


## 4. `pnl_corrections.csv` — corrected execution prices

**Grain:** one row per correction *event* — not per `(position_id,
business_date)`. The same day can be corrected more than once; each row is an
immutable, timestamped fact ("on this date, we were told the execution price
should actually be this"). Nothing here overwrites `daily_pnl.csv` or an
earlier correction row — this file is purely additive.

| Column | Type | Meaning |
|---|---|---|
| `position_id` | string, FK → `positions.position_id` | Which position. |
| `business_date` | date | Which trading day's trade is being corrected. |
| `corrected_execution_price` | number | The new execution price to use in `quantity × (closing_price − execution_price)`, replacing the previous one. |
| `recorded_at` | timestamp | When *this correction* was received/booked — this is the timestamp that determines which month the resulting adjustment belongs to, per the business rules. |

**What's in the sample:** exactly **two correction rows**, both for
**P001 (Alpha Corp), 2025-01-15** — i.e., the same day is corrected twice:

1. `2025-03-05 10:15:00` → execution price corrected to `49.40`
2. `2025-03-12 14:30:00` → execution price corrected to `49.20`

This is the sample's test case for the rule *"a later correction to the same
position, day, and price replaces the earlier correction — it does not add to
it."* Both corrections concern a January day, but land in March (after
January's close on 2025-02-03 — see §5), so both produce **adjustments booked
in March**, not restatements of January.


In [6]:
pnl_corrections = pd.read_csv(DATA_DIR / "pnl_corrections.csv", parse_dates=["business_date", "recorded_at"])
print(pnl_corrections.dtypes, end="\n\n")

# Recompute each correction's implied P&L using the trade's quantity/closing_price,
# paired with the corrected execution price, to see what each correction actually changes to.
merged_corrections = pnl_corrections.merge(
    trade_detail[["position_id", "business_date", "quantity", "closing_price"]],
    on=["position_id", "business_date"],
    how="left",
)
merged_corrections["corrected_pnl"] = merged_corrections["quantity"] * (
    merged_corrections["closing_price"] - merged_corrections["corrected_execution_price"]
)
merged_corrections.sort_values("recorded_at")


position_id                             str
business_date                datetime64[us]
corrected_execution_price           float64
recorded_at                  datetime64[us]
dtype: object



,position_id,business_date,corrected_execution_price,recorded_at,quantity,closing_price,corrected_pnl
0,P001,2025-01-15,49.4,2025-03-05 10:15:00,1000,50.0,600.0
1,P001,2025-01-15,49.2,2025-03-12 14:30:00,1000,50.0,800.0


## 5. `period_status.csv` — when each month closed

**Grain:** one row per calendar month. This is the record of when a reporting
period was closed and sent to investors — after this point, that month's
*reporting-view* figures are frozen.

| Column | Type | Meaning |
|---|---|---|
| `period` | string (`YYYY-MM`) | The month. |
| `closed_at` | timestamp | When that month was closed. |

| period | closed_at |
|---|---|
| 2025-01 | 2025-02-03 09:00:00 |
| 2025-02 | 2025-03-05 09:00:00 |
| 2025-03 | 2025-04-02 09:00:00 |

**Why this matters for the two corrections:** January closed on
**2025-02-03**. Both corrections to the 2025-01-15 trade were received well
after that (2025-03-05 and 2025-03-12) — so January's reporting-view figure
is untouched, and both corrections instead produce adjustments in the month
they were *received* in:

- Correction 1 (`recorded_at` 2025-03-05 10:15) was received *after* February
  closed (2025-03-05 09:00, just ~1 hour earlier) — so it lands in **March**,
  not February. This is a close call worth double-checking carefully against
  the exact timestamps when implementing the "which month does this correction
  belong to" logic.
- Correction 2 (`recorded_at` 2025-03-12 14:30) is comfortably inside March,
  which is still open (closes 2025-04-02) — so it's also a **March** adjustment.


In [7]:
period_status = pd.read_csv(DATA_DIR / "period_status.csv", parse_dates=["closed_at"])
print(period_status.dtypes, end="\n\n")
period_status


period                  str
closed_at    datetime64[us]
dtype: object



,period,closed_at
0,2025-01,2025-02-03 09:00:00
1,2025-02,2025-03-05 09:00:00
2,2025-03,2025-04-02 09:00:00


## 6. How the five files relate

```
positions.csv (1 row per position)
     │ position_id
     ├──────────────────────────────┬───────────────────────────┐
     ▼                               ▼                            ▼
daily_pnl.csv                 trade_detail.csv             pnl_corrections.csv
(1 row per position × day,    (1 row per position × day    (1+ rows per position × day
 original figure only)         that has trade detail —      that was corrected — can
                                here, just the one day       repeat for the same day)
                                that later gets corrected)
     │ business_date                 │ business_date               │ business_date,
     │                                │                             │ recorded_at
     └───────────────┬────────────────┘                             │
                      ▼                                              │
           period_status.csv  ◄───────── recorded_at decides which ──┘
           (1 row per month,             month a correction's adjustment
            closed_at)                   is booked into (NOT business_date's month)
```

Key point for the design: `daily_pnl.csv` and `pnl_corrections.csv` are both
**event logs**, not current-state tables — a given `(position_id,
business_date)` can appear once in `daily_pnl.csv` (the original booking) and
zero, one, or many times in `pnl_corrections.csv` (every correction ever
received for it). The storage structure we design needs to preserve *all* of
these rows and let us ask "what was known as of timestamp T" by filtering on
`recorded_at`.


## 7. A naive baseline rollup (original figures only, no corrections yet)

This is **not** the reporting view or the corrected view — it's just
`sum(pnl_amount)` from `daily_pnl.csv` grouped by position and month, with no
adjustment logic applied at all. It's useful as a sanity-check baseline: once
the restatement logic is built, the *reporting view* for January/February
should equal these numbers exactly (since no correction was received before
their close), and only March's reporting-view total should differ from this
baseline (because both adjustments land in March).


In [8]:
daily_pnl_with_position = daily_pnl.merge(positions, on="position_id", how="left")
daily_pnl_with_position["month"] = daily_pnl_with_position["business_date"].dt.to_period("M").astype(str)

naive_monthly_by_position = (
    daily_pnl_with_position
    .groupby(["security_name", "month"], as_index=False)["pnl_amount"]
    .sum()
    .pivot(index="security_name", columns="month", values="pnl_amount")
)
naive_monthly_by_position["Q1_total"] = naive_monthly_by_position.sum(axis=1)

naive_monthly_fund = daily_pnl_with_position.groupby("month")["pnl_amount"].sum()

print("Naive monthly total by position (pre-correction baseline):")
display(naive_monthly_by_position)

print("\nNaive monthly total for the whole fund (pre-correction baseline):")
display(naive_monthly_fund)


Naive monthly total by position (pre-correction baseline):


month,2025-01,2025-02,2025-03,Q1_total
security_name,,,,
Alpha Corp,3000,2500,1800,7300
Beta Holdings,-1200,900,2200,1900
Gamma Industries,450,-300,700,850



Naive monthly total for the whole fund (pre-correction baseline):


month
2025-01    2250
2025-02    3100
2025-03    4700
Name: pnl_amount, dtype: int64

## 8. Data-quality notes and things to carry into the design

- **No missing values** in any file — every expected column is populated for every row (confirmed via `.isna().sum()` below).
- **No weekend rows** in `daily_pnl.csv` — business dates jump Fri → Mon, consistent with trading days only. Any "every calendar day" assumption in the design would be wrong.
- **`trade_detail.csv` is intentionally sparse** — only 1 of the ~186 position-days in `daily_pnl.csv` has trade detail. The design must not assume every day is recomputable from `quantity × (closing_price − execution_price)`; only corrected days need that recomputation, and trade detail is only supplied for those.
- **Corrections key off `(position_id, business_date)`, not off a trade/row ID** — there's no explicit foreign key back to a specific `daily_pnl.csv` row or `trade_detail.csv` row. The join is purely on position + date, which the "a correction concerns a day" business rule supports.
- **The two corrections are genuinely sequential and both still valid events** — `pnl_corrections.csv` keeps row 1 (`49.40`) even after row 2 (`49.20`) arrives. The "replace, don't add" rule must be implemented as *last-correction-wins by `recorded_at`* for computing the **current** figure, while **both rows remain in storage** untouched.
- **Timestamps matter down to the hour** — the February close (`2025-03-05 09:00`) and the first correction (`2025-03-05 10:15`) are on the same calendar day but ~75 minutes apart. Any logic that compares dates instead of full timestamps would misattribute this correction's adjustment to the wrong month.


In [9]:
print("Missing values per file:\n")
for name, df in [
    ("positions", positions),
    ("daily_pnl", daily_pnl),
    ("trade_detail", trade_detail),
    ("pnl_corrections", pnl_corrections),
    ("period_status", period_status),
]:
    print(f"  {name}: {df.isna().sum().sum()} missing values across {len(df)} rows")

print("\nBusiness-day gaps check (daily_pnl.csv), weekday value counts:")
print(daily_pnl["business_date"].dt.day_name().value_counts())

print("\nDuplicate (position_id, business_date) rows within daily_pnl.csv (should be 0 — one booking per day):")
print(daily_pnl.duplicated(subset=["position_id", "business_date"]).sum())


Missing values per file:

  positions: 0 missing values across 3 rows
  daily_pnl: 0 missing values across 192 rows
  trade_detail: 0 missing values across 1 rows
  pnl_corrections: 0 missing values across 2 rows
  period_status: 0 missing values across 3 rows

Business-day gaps check (daily_pnl.csv), weekday value counts:
business_date
Wednesday    39
Thursday     39
Friday       39
Monday       39
Tuesday      36
Name: count, dtype: int64

Duplicate (position_id, business_date) rows within daily_pnl.csv (should be 0 — one booking per day):
0
